In [ ]:
%load_ext autoreload
%autoreload 2

# áp dụng Heuristic Rules phân loại file không phải native pdf 


In [6]:
import pymupdf

doc = pymupdf.open("../data/raw/resumes/data-ai/1b202ad7-ee8b-460d-b364-0dcb39e4702c.pdf")
page = doc[0]  # Load the first page

# Returns a list of tuples: (x0, y0, x1, y1, "word", block_no, line_no, word_no)
blocks = page.get_text("blocks", sort="True")

for b in blocks:
    bbox = b[:4]
    text = b[4].strip()
    print(f"Block Text: '{text}' Block BBox: {bbox}\n---")


Block Text: 'EXPERIENCES' Block BBox: (314.47918701171875, 55.68933868408203, 432.61614990234375, 82.2432861328125)
---
Block Text: 'TEPA   2021 - now' Block BBox: (404.53851318359375, 96.74492645263672, 497.5958557128906, 112.11329650878906)
---
Block Text: 'Researching and developing Speech to Text' Block BBox: (319.9718933105469, 131.85691833496094, 565.3736572265625, 147.22528076171875)
---
Block Text: 'Vietnamese project.' Block BBox: (319.9718933105469, 149.07931518554688, 432.684814453125, 164.4476776123047)
---
Block Text: 'Researching and developing Text to Speech' Block BBox: (319.9718933105469, 166.30172729492188, 565.3736572265625, 181.6700897216797)
---
Block Text: 'Vietnamese project' Block BBox: (319.9718933105469, 183.5241241455078, 429.59344482421875, 198.89248657226562)
---
Block Text: 'Building Vietnamese CRM Chatbot.' Block BBox: (319.9718933105469, 200.74652099609375, 515.0957641601562, 216.11488342285156)
---
Block Text: 'Building VoiceBot for Cenhomes Sales' Bloc

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load model and tokenizer
model_name = "yashpwr/resume-ner-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Example resume text
# text = "John Smith is a senior software engineer with 8 years of experience at Google. He has expertise in Python, JavaScript, and machine learning. Contact: john.smith@gmail.com"

# Tokenize
inputs = tokenizer(
    markdown_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    padding=True
)

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)

# Extract entities
entities = []
current_entity = None

for i, pred in enumerate(predictions[0]):
    label = model.config.id2label[pred.item()]
    token_id = inputs["input_ids"][0][i].item()
    token = tokenizer.convert_ids_to_tokens(token_id)
    
    if label.startswith('B-'):
        if current_entity:
            entities.append(current_entity)
        current_entity = {
            'text': token,
            'label': label[2:],  # Remove 'B-' prefix
            'start': i
        }
    elif label.startswith('I-') and current_entity:
        current_entity['text'] += ' ' + token
    elif label == 'O':
        if current_entity:
            entities.append(current_entity)
            current_entity = None

if current_entity:
    entities.append(current_entity)

print("Extracted Entities:")
for entity in entities:
    print(f"- {entity['label']}: {entity['text']}")


In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer


def predict_NuExtract(model, tokenizer, text, schema, example=["","",""]):
    schema = json.dumps(json.loads(schema), indent=4)
    input_llm =  "<|input|>\n### Template:\n" +  schema + "\n"
    for i in example:
      if i != "":
          input_llm += "### Example:\n"+ json.dumps(json.loads(i), indent=4)+"\n"
    
    input_llm +=  "### Text:\n"+text +"\n<|output|>\n"
    input_ids = tokenizer(input_llm, return_tensors="pt", truncation=True, max_length=4000).to("cuda")

    output = tokenizer.decode(model.generate(**input_ids)[0], skip_special_tokens=True)
    return output.split("<|output|>")[1].split("<|end-output|>")[0]


model = AutoModelForCausalLM.from_pretrained("numind/NuExtract-tiny", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("numind/NuExtract-tiny", trust_remote_code=True)

model.to("cuda")

model.eval()

text = """We introduce Mistral 7B, a 7–billion-parameter language model engineered for
superior performance and efficiency. Mistral 7B outperforms the best open 13B
model (Llama 2) across all evaluated benchmarks, and the best released 34B
model (Llama 1) in reasoning, mathematics, and code generation. Our model
leverages grouped-query attention (GQA) for faster inference, coupled with sliding
window attention (SWA) to effectively handle sequences of arbitrary length with a
reduced inference cost. We also provide a model fine-tuned to follow instructions,
Mistral 7B – Instruct, that surpasses Llama 2 13B – chat model both on human and
automated benchmarks. Our models are released under the Apache 2.0 license.
Code: https://github.com/mistralai/mistral-src
Webpage: https://mistral.ai/news/announcing-mistral-7b/"""

schema = """{
    "Model": {
        "Name": "",
        "Number of parameters": "",
        "Number of max token": "",
        "Architecture": []
    },
    "Usage": {
        "Use case": [],
        "Licence": ""
    }
}"""

prediction = predict_NuExtract(model, tokenizer, markdown_text, schema, example=["","",""])
print(prediction)


In [ ]:
# reder image
import fitz
from pathlib import Path
import pymupdf 

def pdf_to_image(pdf_path: str, output_dir: str):
    output = Path(output_dir)
    
    doc = pymupdf.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap()
        out_path = output / f"{Path(pdf_path).stem}_page_{page_num+1}.png"
        pix.save(str(out_path))

    doc.close()

pdf_path = "/home/hoai/user/resource/fipilot-be/data/raw/resumes/data-ai/Ai_Engineer_candidate_ai-engineer-9753752.pdf"
save = "/home/hoai/user/resource/fipilot-be/notebooks"
pdf_to_image(pdf_path, save)

In [19]:
import pymupdf  # Also known as fitz

# 1. Open the PDF document
doc = pymupdf.open(pdf_path)

# 2. Iterate through pages
bboxs = []
for page_num in range(len(doc)):
    page = doc[page_num]
    
    # 3. Extract structural text data
    page_dict = page.get_text("dict")
    
    print(f"--- Page {page_num + 1} ---")
    
    # 4. Traverse the structural dictionary hierarchy
    for block in page_dict["blocks"]:
        if "lines" in block:  # Check if the block contains text lines
            for line in block["lines"]:
                # The line bounding box: (x0, y0, x1, y1)
                line_bbox = line["bbox"]
                
                # Combine all spans (text fragments) inside this specific line
                line_text = "".join([span["text"] for span in line["spans"]])
                
                print(f"BBox: {line_bbox}")
                print(f"Text: {line_text}\n")
                bboxs.append(line_bbox)


--- Page 1 ---
BBox: (184.11599731445312, 28.265108108520508, 402.6839904785156, 48.927608489990234)
Text: NGUYỄN MINH DŨNG

BBox: (259.1619873046875, 107.08731842041016, 324.9111328125, 117.99642181396484)
Text: EDUCATION

BBox: (45.16299819946289, 127.96231842041016, 274.8431701660156, 138.8714141845703)
Text: University of Information Technology, VNUHCM

BBox: (465.468994140625, 127.81266784667969, 557.9999389648438, 138.72177124023438)
Text: Aug 2015 - Oct 2019

BBox: (45.162994384765625, 141.36167907714844, 213.52313232421875, 152.27078247070312)
Text: Bachelor’s degree, Computer Science

BBox: (45.162994384765625, 154.9106903076172, 176.93399047851562, 165.81979370117188)
Text: Hornor program, GPA 8.6/10

BBox: (45.162994384765625, 168.45970153808594, 357.5236511230469, 179.6207275390625)
Text: Thesis: Pedestrian Detection Using Cross-modal Deep Representations

BBox: (45.162994384765625, 188.99705505371094, 557.99951171875, 199.90615844726562)
Text: - Implemented the state-of-th

In [26]:
import cv2

img = cv2.imread("/home/hoai/user/resource/fipilot-be/notebooks/Ai_Engineer_candidate_ai-engineer-9753752_page_2.png")
for box in bboxs:
    x0, y0, x1, y1 = map(int, box)

    cv2.rectangle(
        img,
        (x0, y0),
        (x1, y1),
        (0, 255, 0),
        2
    )
cv2.imshow("image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()